# Esfera fluida: energia interna, mapa de estados ressonantes e relações de existência e de recorrência

Notebook ilustrativo do capítulo *Espalhamento por uma partícula esférica fluida* da tese. Cada parte apresenta primeiro as
equações (na mesma notação do texto) e depois o código que as calcula e as figuras correspondentes.

1. Configurações e estética das figuras
2. Modelo: potenciais, condições de contorno e coeficientes
3. Energia interna: fórmulas fechadas das integrais radiais
4. Limite de esfera pequena: ressonância de Minnaert
5. Curvas de energia: ar e petróleo, contrastes de velocidade e de densidade, decomposição modal
6. Banco de dados e contagem das ressonâncias
7. Mapa de estados ressonantes
8. Relação de existência
9. Relação de recorrência
10. Custo computacional
11. Arquivos gerados

As partes 1 a 5 só precisam deste notebook. As partes 6 a 10 leem a saída do gerador da esfera (`gerador_esfera.py`, versão 3):
`resumo_esfera_fluida_lmax10.csv` e `ressonancias_malha_esfera_fluida_lmax10.csv`, na pasta indicada em `PASTA`.

## 1. Configurações
No Colab, a célula abaixo instala o LaTeX usado nos rótulos (só na primeira vez). No PC, basta ter uma distribuição LaTeX com `dvipng` e `cm-super`.

**Onde ficam os resultados.** No Colab, o Google Drive é montado logo no início, e todas as figuras e tabelas são gravadas direto em
`Meu Drive/sphere_final/resultados/`, ao lado dos dados, de modo que nada se perde quando a sessão termina. Fora do Colab, elas vão para
`resultados/`, na pasta do notebook (a mesma estrutura do repositório):

```
resultados/
├── figuras/   png (1200 dpi), pdf e eps de cada figura
└── tabelas/   csv, tex, png, pdf e svg de cada tabela
```

In [ ]:
%%capture
import os, sys, subprocess
if os.path.exists("/content"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq texlive texlive-latex-extra texlive-fonts-recommended dvipng cm-super", shell=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

In [ ]:
import os, glob, time, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, NullFormatter, FuncFormatter
from scipy.special import spherical_jn, spherical_yn
from scipy.integrate import quad
from scipy.spatial import cKDTree
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

USA_LATEX = shutil.which("latex") is not None
plt.rcParams.update({
    "text.usetex": USA_LATEX,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "text.latex.preamble": r"\usepackage{amsmath}",
    "figure.dpi": 150,                      # só a exibição no notebook; os arquivos saem com OUTPUT_DPI
})
VIRG = "{,}" if USA_LATEX else ","          # vírgula decimal ("{,}" evita o espaço extra do LaTeX)
PCT = r"\%" if USA_LATEX else "%"

# estética padrão das figuras da tese
OUTPUT_DPI = 1200
COLORS = ["#6488A5", "#8B1A1A", "black", "0.45"]
LINESTYLES = ["-", "--", "-.", ":"]
LINEWIDTHS = [2.0, 2.0, 1.75, 1.6]

# ---------------- pastas: dados e resultados ----------------
# Google Drive: Meu Drive > sphere_final
# https://drive.google.com/drive/folders/1Bpuzi1Kj9tkbly6Cq3SD0r0R0f3kurYI
try:
    from google.colab import drive
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB:
    drive.mount("/content/drive")
    PASTA = "/content/drive/MyDrive/sphere_final"                     # saída do gerador
    SAIDA = os.path.join(PASTA, "resultados")                          # figuras e tabelas, gravadas no Drive
else:
    # procura os dados em ESFERA_DADOS, depois em dados/ (repositório) e por fim em Downloads/sphere_final
    candidatas = [os.path.abspath("dados"), os.path.join(os.path.expanduser("~"), "Downloads", "sphere_final")]
    PASTA = os.environ.get("ESFERA_DADOS") or next(
        (p for p in candidatas if glob.glob(os.path.join(p, "resumo_*.csv"))), candidatas[0])
    SAIDA = os.path.abspath(os.environ.get("ESFERA_SAIDA", "resultados"))
FIG_DIR = os.path.join(SAIDA, "figuras")
TAB_DIR = os.path.join(SAIDA, "tabelas")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)


def salvar_figura(fig, nome):
    base = os.path.join(FIG_DIR, nome)
    fig.savefig(base + ".png", dpi=OUTPUT_DPI, bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="white")
    fig.savefig(base + ".pdf", bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="white")
    fig.savefig(base + ".eps", format="eps", bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="white")


def dec(valor, casas):
    return f"{valor:.{casas}f}".replace(".", VIRG)


def estilizar(ax, xlabel=None, ylabel=None):
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=12)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=12)
    ax.tick_params(axis="both", which="both", direction="in", length=4, width=0.9, labelsize=10)
    ax.tick_params(axis="both", which="minor", length=2)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_edgecolor("black")
    ax.grid(False)


def adicionar_caixa(ax, texto, x=0.965, y=0.955, ha="right"):
    ax.text(x, y, texto, transform=ax.transAxes, ha=ha, va="top", fontsize=9, linespacing=1.10, zorder=10,
            bbox={"boxstyle": "square,pad=0.30", "facecolor": "white", "edgecolor": "black", "linewidth": 1.0, "alpha": 1.0})


def adicionar_legenda(ax, loc="lower right", **kw):
    leg = ax.legend(loc=loc, fontsize=kw.pop("fontsize", 9), frameon=True, handlelength=2.8, labelspacing=0.30,
                    handletextpad=0.7, **kw)
    leg.get_frame().set(edgecolor="black", facecolor="white", linewidth=1.0, alpha=1.0)
    return leg


print("LaTeX:", USA_LATEX, "| Colab:", NO_COLAB, "\ndados:", PASTA, "\nfiguras:", FIG_DIR, "\ntabelas:", TAB_DIR)

## 2. Modelo

Uma onda plana monocromática, de potencial $\varphi_{\text{inc}}$, incide ao longo de $+z$ sobre uma esfera fluida homogênea de raio $a$,
densidade $\rho_S$ e velocidade do som $c_S$, imersa em um fluido de densidade $\rho_M$ e velocidade do som $c_M$. Em cada meio o
potencial satisfaz a equação de Helmholtz, $(\nabla^2+k^2)\varphi=0$, com $k_M=\omega/c_M$ fora e $k_S=\omega/c_S$ dentro da esfera:

$$
\begin{aligned}
\varphi_{\text{inc}}(k_Mr,\cos\theta) &= A\sum_{\ell=0}^{\infty}(2\ell+1)\,i^\ell j_\ell(k_Mr)P_\ell(\cos\theta),\\
\varphi_{S}(k_Sr,\cos\theta) &= A\sum_{\ell=0}^{\infty}b_\ell\,(2\ell+1)\,i^\ell j_\ell(k_Sr)P_\ell(\cos\theta),\\
\varphi_{\text{sc}}(k_Mr,\cos\theta) &= A\sum_{\ell=0}^{\infty}s_\ell\,(2\ell+1)\,i^\ell h^{(1)}_\ell(k_Mr)P_\ell(\cos\theta).
\end{aligned}
$$

As condições de contorno em $r=a$ impõem a continuidade da pressão e da velocidade normal,

$$
\rho_M(\varphi_{\text{inc}}+\varphi_{\text{sc}})=\rho_S\varphi_S,\qquad
\partial_r(\varphi_{\text{inc}}+\varphi_{\text{sc}})=\partial_r\varphi_S ,
$$

e, usando o wronskiano $j_\ell h_\ell^{(1)\prime}-j_\ell' h_\ell^{(1)} = i/x^2$, fornecem

$$
b_\ell=\frac{i\,x_M^{-2}\,(\rho_M/\rho_S)}{D_\ell},\qquad
s_\ell=-\frac{j_\ell'(x_M)j_\ell(x_S)-m_t\,j_\ell(x_M)j_\ell'(x_S)}{D_\ell},\qquad
D_\ell=h_\ell^{(1)\prime}(x_M)\,j_\ell(x_S)-m_t\,j_\ell'(x_S)\,h_\ell^{(1)}(x_M),
$$

com $x_M=k_Ma$, $x_S=k_Sa=m\,x_M$, $m=c_M/c_S$, $m_t=Z_M/Z_S$ e $\rho=\rho_S/\rho_M$, de modo que $m_t=m/\rho$.
Os zeros de $D_\ell$ ficam no plano complexo, em $x_M=x_k-i\gamma_k$; quando um deles está perto do eixo real, $|b_\ell|$ cresce em torno de
$x_M=x_k$, com largura $2\gamma_k$, e aparece uma ressonância.

As funções de ordem negativa seguem $j_{-n-1}=(-1)^{n+1}y_n$, de modo que $j_{-1}(x)=\cos x/x$.

In [ ]:
def J(n, x):
    x = np.asarray(x, dtype=float)
    if n >= 0:
        return spherical_jn(n, x)
    return (-1) ** (-n) * spherical_yn(-n - 1, x)


def Y(n, x):
    x = np.asarray(x, dtype=float)
    if n >= 0:
        return spherical_yn(n, x)
    return (-1) ** (-n - 1) * spherical_jn(-n - 1, x)


def dJ(n, x):
    x = np.asarray(x, dtype=float)
    if n >= 0:
        return spherical_jn(n, x, derivative=True)
    return (-1) ** (-n) * spherical_yn(-n - 1, x, derivative=True)


def dY(n, x):
    x = np.asarray(x, dtype=float)
    if n >= 0:
        return spherical_yn(n, x, derivative=True)
    return (-1) ** (-n - 1) * spherical_jn(-n - 1, x, derivative=True)


def H(n, x):
    return J(n, x) + 1j * Y(n, x)


def dH(n, x):
    return dJ(n, x) + 1j * dY(n, x)


def D_ell(ell, xM, m, rho_ratio):
    # D_l = h_l'(x_M) j_l(x_S) - m_t j_l'(x_S) h_l(x_M),  x_S = m x_M,  m_t = m/rho
    xS = m * np.asarray(xM, dtype=float)
    return dH(ell, xM) * J(ell, xS) - (m / rho_ratio) * dJ(ell, xS) * H(ell, xM)


def b_ell(ell, xM, m, rho_ratio):
    # b_l = i x_M^-2 (rho_M/rho_S) / D_l
    xM = np.asarray(xM, dtype=float)
    return 1j / (rho_ratio * xM ** 2 * D_ell(ell, xM, m, rho_ratio))


x_teste = 0.73
print("j_{-1}(x) - cos(x)/x =", abs(J(-1, x_teste) - np.cos(x_teste) / x_teste))

## 3. Energia interna

A energia média acumulada no interior da esfera é

$$
W_T=W_P+W_K=\int_V\frac{|p|^2}{4\rho_Sc_S^2}\,dV+\int_V\frac{\rho_S|\mathbf v|^2}{4}\,dV ,
$$

com $p=i\rho_S\omega\varphi_S$ e $\mathbf v=\nabla\varphi_S$. Substituindo $\varphi_S$ e usando a ortogonalidade dos polinômios de Legendre,
as integrais angulares fornecem o fator $4\pi/(2\ell+1)$ e restam integrais radiais que têm forma fechada:

$$
\int_0^{x_S}t^2j_\ell^2(t)\,dt=\frac{x_S^3}{2}\Big[j_\ell^2(x_S)-j_{\ell-1}(x_S)\,j_{\ell+1}(x_S)\Big]\equiv F_\ell(x_S),
\qquad
\int_0^{x_S}\Big[t^2j_\ell'^2(t)+\ell(\ell+1)j_\ell^2(t)\Big]dt=F_\ell(x_S)+x_S^2\,j_\ell(x_S)\,j_\ell'(x_S).
$$

Com a normalização $2f_0/(Z_S|A|^2)$, em que $f_0$ é a frequência da onda incidente,

$$
\frac{2f_0W_P}{Z_S|A|^2}=\sum_{\ell=0}^{\infty}|b_\ell|^2(2\ell+1)\,F_\ell(x_S),
\qquad
\frac{2f_0W_K}{Z_S|A|^2}=\sum_{\ell=0}^{\infty}|b_\ell|^2(2\ell+1)\Big[F_\ell(x_S)+x_S^2\,j_\ell(x_S)\,j_\ell'(x_S)\Big].
$$

Só a soma em $\ell$ precisa ser truncada. O critério
$\ell_{\max}=3+\lfloor x_M^{(\max)}+4{,}05\,(x_M^{(\max)})^{1/3}\rfloor$ dá $\ell_{\max}=8$ para $x_M\le1$, e adota-se $\ell_{\max}=10$.
A célula seguinte confere as duas fórmulas fechadas contra a integração numérica.

In [ ]:
def F_ell(ell, xS):
    # F_l(x_S) = (x_S^3/2) [ j_l^2(x_S) - j_{l-1}(x_S) j_{l+1}(x_S) ]
    return 0.5 * xS ** 3 * (J(ell, xS) ** 2 - J(ell - 1, xS) * J(ell + 1, xS))


def B_ell(ell, xS):
    # termo de borda da parcela cinética: x_S^2 j_l(x_S) j_l'(x_S)
    return xS ** 2 * J(ell, xS) * dJ(ell, xS)


def energia_modal(xM, m, rho_ratio, ell):
    # W_T^(l), W_P^(l), W_K^(l) normalizadas por 2 f_0/(Z_S |A|^2)
    xM = np.asarray(xM, dtype=float)
    xS = m * xM
    with np.errstate(all="ignore"):
        peso = (2 * ell + 1) * np.abs(b_ell(ell, xM, m, rho_ratio)) ** 2
        WP = np.real(peso * F_ell(ell, xS))
        WK = np.real(peso * (F_ell(ell, xS) + B_ell(ell, xS)))
    WP[~np.isfinite(WP)] = np.nan
    WK[~np.isfinite(WK)] = np.nan
    return WP + WK, WP, WK


def energia_total(xM, m, rho_ratio, lmax=10):
    WT = WP = WK = 0.0
    for ell in range(lmax + 1):
        t, p, k = energia_modal(xM, m, rho_ratio, ell)
        WT, WP, WK = WT + t, WP + p, WK + k
    return WT, WP, WK


print(" l    x_S    integral numérica    fórmula fechada (potencial)    integral numérica    fórmula fechada (cinética)")
for ell in (0, 1, 3, 6):
    for xS in (0.5, 4.0, 12.0):
        num_P = quad(lambda t: t ** 2 * J(ell, t) ** 2, 0.0, xS, limit=200)[0]
        num_K = quad(lambda t: t ** 2 * dJ(ell, t) ** 2 + ell * (ell + 1) * J(ell, t) ** 2, 0.0, xS, limit=200)[0]
        print(f"{ell:2d} {xS:6.1f} {num_P:18.10e} {F_ell(ell, xS):22.10e} {num_K:28.10e} {F_ell(ell, xS) + B_ell(ell, xS):22.10e}")

## 4. Limite de esfera pequena

Para $x_M\ll1$ e $x_S\ll1$, só o termo $\ell=0$ é ressonante, e as parcelas normalizadas da energia ficam

$$
\frac{2f_0W_K}{Z_S|A|^2}=\frac{m^5x_M^5}{45}\left[\frac{m_t}{m}\frac{3}{3-m_tm\,x_M^2}\right]^2,
\qquad
\frac{2f_0W_P}{Z_S|A|^2}=\frac{4m^3x_M^3}{3}\left[\frac{m_t}{m}\frac{3}{3-m_tm\,x_M^2}\right]^2 .
$$

O denominador se anula em

$$
x_{\mathrm{ref}}=\sqrt{\frac{3}{m\,m_t}}=\frac{\sqrt{3\rho}}{m}=\sqrt{\frac{3\kappa_M}{\kappa_S}},
$$

a ressonância de Minnaert, em que $\kappa=1/(\rho c^2)$ é a compressibilidade adiabática e $m\,m_t=\kappa_S/\kappa_M$. A primeira ressonância só aparece
em frequências baixas quando a esfera é muito mais compressível que o meio. Para a bolha de ar em água do mar ($c_M=1522$ m/s,
$\rho_M=1024$ kg/m$^3$, $c_S=343$ m/s, $\rho_S=1{,}225$ kg/m$^3$):

In [ ]:
def x_ref(m, rho_ratio):
    # x_ref = sqrt(3/(m m_t)) = sqrt(3 rho)/m
    return np.sqrt(3.0 * rho_ratio) / m


m_ar, rho_ar = 1522.0 / 343.0, 1.225 / 1024.0
print(f"m = {m_ar:.3f}, m_t = {m_ar / rho_ar:.4g}, kappa_S/kappa_M = {m_ar ** 2 / rho_ar:.4g}, x_ref = {x_ref(m_ar, rho_ar):.5f}")

## 5. Curvas de energia

### 5.1 Ar e petróleo em água do mar

In [ ]:
X_MIN, X_MAX, N_POINTS, LMAX = 1e-4, 0.10, 30000, 10
C_M, RHO_M = 1522.0, 1024.0                       # água do mar
M_AIR, RHO_RATIO_AIR = C_M / 343.0, 1.225 / RHO_M  # ar
M_OIL, RHO_RATIO_OIL = 1.10, 0.90                  # petróleo

xM = np.geomspace(X_MIN, X_MAX, N_POINTS)
WT_air = energia_total(xM, M_AIR, RHO_RATIO_AIR, LMAX)[0]
WT_oil = energia_total(xM, M_OIL, RHO_RATIO_OIL, LMAX)[0]

fig, ax = plt.subplots(figsize=(5, 3.5))
for WT, cor, ls, lw, rot in ((WT_air, "#8B1A1A", "--", 2.0, r"$Z_S^{\text{Ar}}$"),
                             (WT_oil, "black", "-", 1.75, r"$Z_S^{\text{Petr\'oleo}}$")):
    ok = np.isfinite(WT) & (WT > 0.0)
    ax.plot(xM[ok], WT[ok], color=cor, linestyle=ls, linewidth=lw, label=rot)
ax.set_yscale("log")
ax.set_xlim(0.0, X_MAX)
ax.set_ylim(1e-10, 1e10)
ax.xaxis.set_major_locator(FixedLocator([0.0, 0.025, 0.05, 0.075, 0.1]))
ax.set_xticklabels([r"$0$", r"$0{,}025$", r"$0{,}050$", r"$0{,}075$", r"$0{,}100$"])
ax.yaxis.set_major_locator(FixedLocator([1e-10, 1e-5, 1e0, 1e5, 1e10]))
ax.minorticks_off()
ax.yaxis.set_minor_formatter(NullFormatter())
estilizar(ax, r"$x_M=k_Ma$", r"Energia interna $(u.a.)$")
adicionar_legenda(ax)
adicionar_caixa(ax, r"$Z_M^{\text{\'Agua do mar}}$")
fig.subplots_adjust(left=0.16, right=0.985, bottom=0.15, top=0.98)
salvar_figura(fig, "sphere_air_oil")
plt.show()
plt.close(fig)

### 5.2 Contraste de velocidade e contraste de densidade

In [ ]:
X_MIN, X_MAX, N_POINTS = 1e-4, 0.1, 5000
xM = np.geomspace(X_MIN, X_MAX, N_POINTS)


def eixo_energia(ax):
    ax.set_yscale("log")
    ax.set_xlim(0.0, X_MAX)
    ax.set_ylim(1e-10, 1e10)
    ax.xaxis.set_major_locator(FixedLocator([0.0, 0.025, 0.05, 0.075, 0.1]))
    ax.set_xticklabels([r"$0$", r"$0{,}025$", r"$0{,}050$", r"$0{,}075$", r"$0{,}100$"])
    ax.yaxis.set_major_locator(FixedLocator([1e-10, 1e-5, 1e0, 1e5, 1e10]))
    ax.set_yticklabels([r"$10^{-10}$", r"$10^{-5}$", r"$10^{0}$", r"$10^{5}$", r"$10^{10}$"])
    ax.minorticks_off()
    ax.yaxis.set_minor_formatter(NullFormatter())
    estilizar(ax, r"$x_M=k_Ma$", r"Energia interna $(u.a.)$")


casos = {
    "sphere_energy_variacao_velocidade": dict(
        variavel="m", valores=[10.0, 50.0, 75.0], fixo=1.10,
        rotulos=[r"$c_M/c_S=10$", r"$c_M/c_S=50$", r"$c_M/c_S=75$"], caixa=r"$\rho_S/\rho_M=1{,}10$"),
    "sphere_energy_variacao_densidade": dict(
        variavel="rho", valores=[1 / 10, 1 / 1e3, 1 / 1e4], fixo=1.10,
        rotulos=[r"$\rho_M/\rho_S=10$", r"$\rho_M/\rho_S=10^3$", r"$\rho_M/\rho_S=10^4$"], caixa=r"$c_M/c_S=1{,}10$"),
}
for nome, caso in casos.items():
    fig = plt.figure(figsize=(5.0, 3.5))
    ax = fig.add_subplot(111)
    for i, (valor, rot) in enumerate(zip(caso["valores"], caso["rotulos"])):
        m, rho = (valor, caso["fixo"]) if caso["variavel"] == "m" else (caso["fixo"], valor)
        WT = energia_total(xM, m, rho, LMAX)[0]
        ok = np.isfinite(WT) & (WT > 0.0)
        ax.plot(xM[ok], WT[ok], color=COLORS[i], linestyle=LINESTYLES[i], linewidth=[2.0, 2.0, 1.75][i], label=rot)
    eixo_energia(ax)
    adicionar_caixa(ax, caso["caixa"], y=0.94)
    adicionar_legenda(ax, borderpad=0.45)
    fig.subplots_adjust(left=0.14, right=0.985, bottom=0.13, top=0.98)
    salvar_figura(fig, nome)
    plt.show()
    plt.close(fig)

### 5.3 Decomposição modal

Energia potencial $W_P^{(\ell)}$ (painéis superiores) e cinética $W_K^{(\ell)}$ (painéis inferiores) das ordens $\ell=0,1,2,3$,
primeiro com contraste de densidade e depois com contraste de velocidade.

In [ ]:
def figura_modal(casos, x_max, x_ticks, x_ticklabels, y_ticks, ylim, nome):
    x_low = np.geomspace(1e-5, 0.02, 4000, endpoint=False)
    x_high = np.linspace(0.02, x_max, 8000)
    xM = np.unique(np.concatenate([x_low, x_high]))
    fig, axes = plt.subplots(2, 2, figsize=(10, 7))
    for col, caso in enumerate(casos):
        axP, axK = axes[0, col], axes[1, col]
        for i, ell in enumerate([0, 1, 2, 3]):
            _, WP, WK = energia_modal(xM, caso["m"], caso["rho"], ell)
            for ax, W in ((axP, WP), (axK, WK)):
                ok = np.isfinite(W) & (W > 0.0)
                ax.plot(xM[ok], W[ok], color=COLORS[i], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i], label=rf"$\ell={ell}$")
        for ax, rot in ((axP, r"Energia interna potencial $(u.a.)$"), (axK, r"Energia interna cinética $(u.a.)$")):
            ax.set_yscale("log")
            ax.set_xlim(0.0, x_max)
            ax.set_ylim(*ylim)
            ax.xaxis.set_major_locator(FixedLocator(x_ticks))
            ax.set_xticklabels(x_ticklabels)
            ax.yaxis.set_major_locator(FixedLocator(y_ticks))
            ax.minorticks_off()
            ax.yaxis.set_minor_formatter(NullFormatter())
            estilizar(ax, r"$x_M=k_Ma$", rot)
            adicionar_caixa(ax, caso["caixa"])
            adicionar_legenda(ax)
    fig.subplots_adjust(left=0.09, right=0.985, bottom=0.09, top=0.98, wspace=0.25, hspace=0.27)
    salvar_figura(fig, nome)
    plt.show()
    plt.close(fig)


figura_modal(
    [dict(m=1.10, rho=1e-3, caixa=r"$\rho_M/\rho_S=10^3$" + "\n" + r"$c_M/c_S=1{,}10$"),
     dict(m=1.10, rho=1e-4, caixa=r"$\rho_M/\rho_S=10^4$" + "\n" + r"$c_M/c_S=1{,}10$")],
    0.50, [0.0, 0.125, 0.25, 0.375, 0.5], [r"$0$", r"$0{,}125$", r"$0{,}250$", r"$0{,}375$", r"$0{,}500$"],
    [1e-12, 1e-9, 1e-6, 1e-3, 1e0, 1e3, 1e6], (1e-12, 1e7), "sphere_modal_potential_kinetic_2x2")

figura_modal(
    [dict(m=50.0, rho=0.90, caixa=r"$c_M/c_S=50$" + "\n" + r"$\rho_S/\rho_M=0{,}90$"),
     dict(m=100.0, rho=0.90, caixa=r"$c_M/c_S=100$" + "\n" + r"$\rho_S/\rho_M=0{,}90$")],
    0.20, [0.0, 0.05, 0.1, 0.15, 0.2], [r"$0$", r"$0{,}05$", r"$0{,}10$", r"$0{,}15$", r"$0{,}20$"],
    [1e-15, 1e-10, 1e-5, 1e0, 1e5, 1e10], (1e-15, 1e10), "sphere_modal_velocity_contrast_2x2")

## 6. Banco de dados e contagem das ressonâncias

O gerador calcula $W_T$ para $4\,599$ pares $(m,\rho)$: $63$ valores de $m=c_M/c_S$ entre $10^{-3}$ e $3\times10^3$ e $73$ valores de
$\rho=\rho_S/\rho_M$ entre $10^{-4}$ e $3\times10^3$, com $10^{-4}\le x_M\le1$ e $0\le\ell\le10$.

Uma ressonância é um **pico observado em $W_T$**: um máximo cuja altura supera o fundo por pelo menos $0{,}1$ em $\log_{10}W_T$.
A malha base tem $5\,000$ pontos ($1\,000$ logarítmicos em $[10^{-4};0{,}1]$, $3\,000$ lineares em $[0{,}1;0{,}9]$ e $1\,000$ lineares em
$[0{,}9;1]$). Para $m>1$ a malha é refinada em torno dos zeros de $D_\ell$, até $2\times10^6$ pontos por curva; os zeros indicam
onde refinar, mas a contagem é sempre a dos picos de $W_T$ (`ressonancias_malha_*.csv`). Acima de $m\approx20$ o teto de pontos é atingido e a
contagem passa a ser um limite inferior.

In [ ]:
# PASTA (saída do gerador: resumo_*.csv e ressonancias_malha_*.csv) foi definida na Seção 1


def achar(padrao):
    arqs = sorted(glob.glob(os.path.join(PASTA, "**", padrao), recursive=True))
    return arqs[0] if arqs else None


ARQ_RESUMO = achar("resumo_*.csv")
ARQ_RES = achar("ressonancias_malha_*.csv")
print("resumo:", ARQ_RESUMO)
print("picos de W_T:", ARQ_RES)

resumo = pd.read_csv(ARQ_RESUMO)
resumo = resumo.loc[resumo["status"] == "ok"].sort_values("global").reset_index(drop=True)
resumo["c_razao"] = 1.0 / resumo["m"]                         # c_S/c_M
resumo["u"] = np.log(resumo["m"])                              # ln m
resumo["v"] = np.log(resumo["rho_ratio"])                      # ln rho
resumo["N"] = np.where(resumo["curva_risco"].astype(bool), resumo["N_res_malha_refinada"],
                       resumo["N_res_malha_base"]).astype(int)  # picos observados em W_T
resumo["ressonante"] = (resumo["N"] > 0).astype(int)

print(f"\ncurvas: {len(resumo)} | c_S/c_M de {resumo['c_razao'].min():.3g} a {resumo['c_razao'].max():.4g} | "
      f"rho_S/rho_M de {resumo['rho_ratio'].min():.3g} a {resumo['rho_ratio'].max():.4g}")
print(f"picos de W_T: {resumo['N'].sum()} | curvas com pelo menos um: {resumo['ressonante'].sum()} "
      f"({100 * resumo['ressonante'].mean():.1f} %) | máximo por curva: {resumo['N'].max()}")
print(f"tempo médio do gerador por curva: {resumo['runtime_s'].mean():.2f} s (mediana {resumo['runtime_s'].median():.2f} s)")

r = resumo.loc[resumo["curva_risco"].astype(bool)].groupby("m").agg(
    picos_WT=("N_res_malha_refinada", "median"), polos=("N_polos_visiveis", "median"))
r["fracao"] = r["picos_WT"] / r["polos"]
print("\nfração dos modos resolvida em W_T (mediana sobre rho):")
print(r.round(3).to_string())

## 7. Mapa de estados ressonantes

Número de picos de $W_T$ até $x_M=1$ no plano $(c_S/c_M,\ \rho_S/\rho_M)$, sempre com a esfera dividida pelo meio externo, em
escala logarítmica. A estética é a do mapa $Z_A\times Z_B$ do artigo do core-shell: cada célula mostra o maior número de
ressonâncias entre as curvas que ela contém, e células vazias recebem o valor do vizinho mais próximo. Caixas vermelhas: bolha de ar em água (ar entre $0$ e $30\,^\circ$C,
$1{,}16$ a $1{,}29$ kg/m$^3$ e $331$ a $349$ m/s) e bolha de óleo em água (azeite, $910$ kg/m$^3$ e $1450$ m/s; óleo de
silicone, $950$ kg/m$^3$ e $1100$ m/s). Meio externo: água, $997$ kg/m$^3$ e $1497$ m/s.
A fronteira $P_{\mathrm{res}}=0{,}5$ vem da relação de existência (Seção 8), por isso o mapa é desenhado depois do ajuste.

In [ ]:
# ---------------- estética do mapa Z_A x Z_B (figuras_artigo.ipynb) ----------------
LOG_X = (-3.5, 3.0)                     # log10(c_S/c_M), de 3e-4 a 1000
LOG_Y = (-4.0, 3.5)                     # log10(rho_S/rho_M), de 1e-4 a 3000
LARGURA_BIN = 0.25
NX = int(round((LOG_X[1] - LOG_X[0]) / LARGURA_BIN))
NY = int(round((LOG_Y[1] - LOG_Y[0]) / LARGURA_BIN))
BOX_COLOR = "#600B00"
CELL_LINECOLOR, CELL_LINEWIDTH = "white", 0.42
FRONT_OBS = dict(color="#7FA6D9", linestyle=(0, (5.0, 2.2)), linewidth=2.0, alpha=0.95)
FRONT_LOG = dict(color="#600B00", linestyle=(0, (4.5, 2.5)), linewidth=1.9, alpha=0.92)
RESONANCE_CMAP = LinearSegmentedColormap.from_list("resonance_count", ["white", "#D9E1EA", "#526F92", "black"], N=256)
RESONANCE_CMAP.set_bad("#D9D9D9")
C_AGUA, RHO_AGUA = 1497.0, 997.0          # meio externo: água a 25 °C
COR_CAIXA = "#8B1A1A"                     # vermelho da paleta COLORS
AR = dict(c=(331 / C_AGUA, 349 / C_AGUA), rho=(1.16 / RHO_AGUA, 1.29 / RHO_AGUA))
OLEO = dict(c=(1100 / C_AGUA, 1450 / C_AGUA), rho=(910 / RHO_AGUA, 950 / RHO_AGUA))
LADO_MIN_CAIXA = 0.35                   # lado mínimo de cada caixinha, em células


def celula_x(v):
    return (np.log10(v) - LOG_X[0]) / LARGURA_BIN - 0.5


def celula_y(v):
    return (np.log10(v) - LOG_Y[0]) / LARGURA_BIN - 0.5


def mapa_binado(df):
    ex, ey = np.linspace(*LOG_X, NX + 1), np.linspace(*LOG_Y, NY + 1)
    lx, ly, n = np.log10(df["c_razao"].to_numpy()), np.log10(df["rho_ratio"].to_numpy()), df["N"].to_numpy(float)
    ix = np.clip(np.searchsorted(ex, lx, side="right") - 1, 0, NX - 1)
    iy = np.clip(np.searchsorted(ey, ly, side="right") - 1, 0, NY - 1)
    contagem, maximo = np.zeros((NX, NY), int), np.full((NX, NY), -np.inf)
    np.add.at(contagem, (ix, iy), 1)
    np.maximum.at(maximo, (ix, iy), n)
    vazio = contagem == 0
    if vazio.any():
        ga, gb = np.meshgrid(0.5 * (ex[:-1] + ex[1:]), 0.5 * (ey[:-1] + ey[1:]), indexing="ij")
        _, viz = cKDTree(np.column_stack([lx, ly])).query(np.column_stack([ga[vazio], gb[vazio]]))
        maximo[vazio] = n[viz]
    return np.maximum(maximo, 0.0), contagem


def segmentos_fronteira(mask):
    seg = []
    for i in range(mask.shape[0]):
        for j in range(mask.shape[1]):
            if not mask[i, j]:
                continue
            xl, xr, yb, yt = i - 0.5, i + 0.5, j - 0.5, j + 0.5
            if i == 0 or not mask[i - 1, j]: seg.append(((xl, yb), (xl, yt)))
            if i == mask.shape[0] - 1 or not mask[i + 1, j]: seg.append(((xr, yb), (xr, yt)))
            if j == 0 or not mask[i, j - 1]: seg.append(((xl, yb), (xr, yb)))
            if j == mask.shape[1] - 1 or not mask[i, j + 1]: seg.append(((xl, yt), (xr, yt)))
    return seg


def caixinha(ax, fx, fy, rotulo, lado):
    x0, x1, y0, y1 = celula_x(fx[0]), celula_x(fx[1]), celula_y(fy[0]), celula_y(fy[1])
    if x1 - x0 < LADO_MIN_CAIXA:
        xc = 0.5 * (x0 + x1); x0, x1 = xc - LADO_MIN_CAIXA / 2, xc + LADO_MIN_CAIXA / 2
    if y1 - y0 < LADO_MIN_CAIXA:
        yc = 0.5 * (y0 + y1); y0, y1 = yc - LADO_MIN_CAIXA / 2, yc + LADO_MIN_CAIXA / 2
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=COR_CAIXA, linewidth=1.8, zorder=30))
    caixa = dict(facecolor="white", edgecolor="none", alpha=0.85, boxstyle="square,pad=0.05")
    if lado == "acima":
        ax.text(x1 + 0.12, y1, rotulo, color=COR_CAIXA, fontsize=15, ha="left", va="bottom", zorder=31, bbox=caixa)
    else:
        ax.text(x0 - 0.12, y0, rotulo, color=COR_CAIXA, fontsize=15, ha="right", va="top", zorder=31, bbox=caixa)


def plot_mapa_fase(df, coef_exist, idioma):
    pt = idioma == "pt"
    sep = VIRG if pt else "."
    idx = "S" if pt else "A"                  # tese: esfera S; artigo: esfera A
    mapa, contagem = mapa_binado(df)
    n_max = int(np.ceil(mapa.max()))
    norm = Normalize(vmin=0.0, vmax=np.log10(1.0 + n_max))
    ticks = np.unique(np.array([0] + [10 ** k for k in range(1, 6) if 10 ** k < 0.8 * n_max] + [n_max], dtype=int))
    with plt.rc_context({"font.family": "sans-serif"}):
        altura = 0.70
        largura = altura * NX / NY
        fig = plt.figure(figsize=(10.5, 10.5))
        ax = fig.add_axes([0.12, 0.13, largura, altura])
        cax = fig.add_axes([0.12 + largura + 0.02, 0.13, 0.03, altura])
        for i in range(NX):
            for j in range(NY):
                ax.add_patch(Rectangle((i - 0.5, j - 0.5), 1.0, 1.0, facecolor=RESONANCE_CMAP(norm(np.log10(1.0 + mapa[i, j]))),
                                       edgecolor=CELL_LINECOLOR, linewidth=CELL_LINEWIDTH, zorder=1))
        for a, b in segmentos_fronteira(mapa > 0):
            ax.plot([a[0], b[0]], [a[1], b[1]], solid_capstyle="butt", zorder=17, **FRONT_OBS)
        GX, GY = np.meshgrid(np.linspace(*LOG_X, 600), np.linspace(*LOG_Y, 600))
        G = (termos_existencia(-GX.ravel() * np.log(10), GY.ravel() * np.log(10)) @ coef_exist).reshape(GX.shape)
        ax.contour(celula_x(10 ** GX), celula_y(10 ** GY), G, levels=[0.0], colors=[FRONT_LOG["color"]],
                   linewidths=FRONT_LOG["linewidth"], linestyles=[FRONT_LOG["linestyle"]], alpha=FRONT_LOG["alpha"], zorder=18)
        caixinha(ax, AR["c"], AR["rho"], "bolha de ar\nem água" if pt else "air bubble\nin water", "acima")
        caixinha(ax, OLEO["c"], OLEO["rho"], "bolha de óleo\nem água" if pt else "oil bubble\nin water", "acima")
        fmt = lambda t: f"{t:.4f}".rstrip("0").rstrip(".").replace(".", sep) if t < 1 else f"{t:g}"
        tx = 10.0 ** np.arange(np.ceil(LOG_X[0]), LOG_X[1] + 1e-9)
        ty = 10.0 ** np.arange(np.ceil(LOG_Y[0]), LOG_Y[1] + 1e-9)
        ax.set_xticks(celula_x(tx)); ax.set_xticklabels([fmt(t) for t in tx])
        ax.set_yticks(celula_y(ty)); ax.set_yticklabels([fmt(t) for t in ty])
        ax.set_xlabel(rf"$c_{idx}/c_M$", fontsize=23, labelpad=9)
        ax.set_ylabel(rf"$\rho_{idx}/\rho_M$", fontsize=23, labelpad=10)
        ax.set_xlim(-0.5, NX - 0.5); ax.set_ylim(-0.5, NY - 0.5)
        ax.set_aspect("equal", adjustable="box")
        handles = [Line2D([0], [0], label="Fronteira observada" if pt else "Observed frontier", **FRONT_OBS),
                   Line2D([0], [0], label=("Fronteira logística\n" + r"($P_{\mathrm{res}}=0" + sep + "5$)") if pt
                          else "Logistic frontier\n" + r"($P_{\mathrm{res}}=0.5$)", **FRONT_LOG)]
        leg = ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.985, 0.85), bbox_transform=ax.transAxes,
                        fontsize=20.0, frameon=True, framealpha=1.0, facecolor="white", edgecolor="none", fancybox=False,
                        borderpad=0.20, handlelength=2.4, handletextpad=0.8, labelspacing=0.45, borderaxespad=0.0)
        leg.set_zorder(100)
        ax.text(0.955, 0.965, r"$N_{\mathrm{res}}$", transform=ax.transAxes, fontsize=24, color="white", va="top", ha="right",
                bbox=dict(facecolor=BOX_COLOR, edgecolor="none", alpha=1.0, boxstyle="square,pad=0.14"), zorder=111)
        mapper = cm.ScalarMappable(norm=norm, cmap=RESONANCE_CMAP)
        mapper.set_array([])
        cbar = fig.colorbar(mapper, cax=cax, ticks=np.log10(1.0 + ticks))
        cbar.ax.set_yticklabels([str(t) for t in ticks])
        cbar.set_label("Ressonâncias" if pt else "Resonances", fontsize=20, labelpad=18, rotation=270, color="black")
        for eixo in (ax, cbar.ax):
            eixo.tick_params(axis="both", labelsize=15, width=1.25, length=4.5, direction="out")
            for spine in eixo.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor(BOX_COLOR)
                spine.set_linewidth(1.8)
        nome = f"mapa_fase_esfera_{idioma}"
        for ext in ("png", "pdf", "eps"):
            fig.savefig(os.path.join(FIG_DIR, f"{nome}.{ext}"), dpi=600, bbox_inches="tight")
        plt.show()
        plt.close(fig)

## 8. Relação de existência

$$
P_{\mathrm{res}}(m,\rho)=\frac{1}{1+\exp[-g(m,\rho)]},\qquad
g(m,\rho)=c_0+c_1\ln m+c_2\ln\rho+c_3(\ln m)^2+c_4\ln m\,\ln\rho+c_5(\ln\rho)^2 ,
$$

com $m=c_M/c_S$ e $\rho=\rho_S/\rho_M$ (variáveis do texto). Há ressonância em $10^{-4}\le x_M\le1$ quando $P_{\mathrm{res}}\ge1/2$, isto é,
$g\ge0$. O ajuste usa regressão logística com termos quadráticos; $20\,\%$ das curvas, sorteadas uma vez (semente 42), ficam para o teste,
e são as mesmas curvas de teste da recorrência. As incertezas vêm da informação de Fisher.

In [ ]:
TEST_FRACTION, RANDOM_SEED = 0.20, 42
TEX_TERMOS = [r"$1$", r"$\ln m$", r"$\ln\rho$", r"$(\ln m)^2$", r"$\ln m\,\ln\rho$", r"$(\ln\rho)^2$"]


def termos_existencia(u, v):
    # [1, ln m, ln rho, (ln m)^2, ln m ln rho, (ln rho)^2]
    u, v = np.asarray(u, float), np.asarray(v, float)
    return np.column_stack([np.ones_like(u), u, v, u * u, u * v, v * v])


def p_res(u, v, coef):
    return 1.0 / (1.0 + np.exp(-np.clip(termos_existencia(u, v) @ coef, -700, 700)))


rng = np.random.default_rng(RANDOM_SEED)
indices = np.arange(len(resumo))
rng.shuffle(indices)
n_teste = int(round(TEST_FRACTION * len(indices)))
treino_c, teste_c = resumo.iloc[indices[n_teste:]].copy(), resumo.iloc[indices[:n_teste]].copy()
GLOBAIS_TESTE = set(teste_c["global"])

modelo = Pipeline([("poly", PolynomialFeatures(degree=2, include_bias=False)), ("scale", StandardScaler()),
                   ("clf", LogisticRegression(max_iter=5000))]).fit(treino_c[["u", "v"]].to_numpy(), treino_c["ressonante"].to_numpy())
esc, lr = modelo.named_steps["scale"], modelo.named_steps["clf"]
COEF_EXIST = np.r_[lr.intercept_[0] - np.sum(lr.coef_[0] * esc.mean_ / esc.scale_), lr.coef_[0] / esc.scale_]
X = termos_existencia(treino_c["u"], treino_c["v"])
p = p_res(treino_c["u"], treino_c["v"], COEF_EXIST)
ERRO_EXIST = np.sqrt(np.clip(np.diag(np.linalg.pinv(X.T @ (X * (p * (1 - p))[:, None]))), 0, None))


def metricas_existencia(df):
    y, p = df["ressonante"].to_numpy(), p_res(df["u"], df["v"], COEF_EXIST)
    pred = (p >= 0.5).astype(int)
    return {"N": len(y), "fracao": y.mean(), "acuracia": accuracy_score(y, pred), "precisao": precision_score(y, pred),
            "revocacao": recall_score(y, pred), "F1": f1_score(y, pred), "AUC": roc_auc_score(y, p),
            "FP": int(((pred == 1) & (y == 0)).sum()), "FN": int(((pred == 0) & (y == 1)).sum())}


MET_EXIST = {"treino": metricas_existencia(treino_c), "teste": metricas_existencia(teste_c)}
for i, t in enumerate(["1", "ln m", "ln rho", "(ln m)^2", "ln m ln rho", "(ln rho)^2"]):
    aviso = "" if abs(COEF_EXIST[i]) >= 2 * ERRO_EXIST[i] else "   <- |c| < 2 sigma (mantido na fórmula)"
    print(f"c{i} ({t:>12s}) = {COEF_EXIST[i]:+.5e} ± {ERRO_EXIST[i]:.2e}{aviso}")
for parte, m_ in MET_EXIST.items():
    print(parte, {k: round(v, 4) if isinstance(v, float) else v for k, v in m_.items()})

### Tabelas de coeficientes

Mesmo formato das tabelas da tese: valor $\pm$ incerteza com dois algarismos significativos, em PNG, PDF, SVG, CSV e `.tex`.

In [ ]:
def fmt_pm(valor, erro):
    if not (np.isfinite(erro) and erro > 0.0):
        return f"${dec(valor, 4)}$"
    casas = max(0, -int(np.floor(np.log10(erro))) + 1)
    return rf"${f'{valor:+.{casas}f}'.replace('.', VIRG)}\pm{f'{erro:.{casas}f}'.replace('.', VIRG)}$"


def save_compact_table(base, dados, linhas, colunas, figsize, first_col_width=0.16, header_label="Coef.", fontsize=17):
    with plt.rc_context({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 30}):
        fig, ax = plt.subplots(figsize=figsize, facecolor="white")
        ax.axis("off")
        tab = ax.table(cellText=[[linhas[i], *dados[i]] for i in range(len(linhas))], colLabels=[header_label, *colunas],
                       cellLoc="center", colWidths=[first_col_width] + [(1 - first_col_width) / len(colunas)] * len(colunas),
                       bbox=[0.02, 0.02, 0.96, 0.96])
        tab.auto_set_font_size(False)
        tab.set_fontsize(fontsize)
        for (lin, col), cel in tab.get_celld().items():
            cel.visible_edges = ""
            cel.set_linewidth(0.0)
            cel.PAD = 0.01
            cel.get_text().set_ha("left" if col == 0 else "center")
            cel.get_text().set_linespacing(0.90)
        for col in range(len(colunas) + 1):
            tab[(0, col)].visible_edges = "TB"
            tab[(0, col)].set_linewidth(1.5)
            tab[(0, col)].get_text().set_fontsize(fontsize + 1)
            tab[(len(linhas), col)].visible_edges = "B"
            tab[(len(linhas), col)].set_linewidth(1.5)
        tab[(0, 0)].get_text().set_fontweight("bold")
        fig.subplots_adjust(left=0.005, right=0.995, bottom=0.005, top=0.995)
        for ext in ("png", "pdf", "svg"):
            fig.savefig(f"{base}.{ext}", dpi=600, bbox_inches="tight", pad_inches=0.01, facecolor="white")
        plt.show()
        plt.close(fig)


def salvar_tabela_coeficientes(nome, linhas, colunas, valores, erros, figsize, first_col_width=0.16, fontsize=17):
    base = os.path.join(TAB_DIR, nome)
    dados = [[fmt_pm(valores[i][j], erros[i][j]) for j in range(len(colunas))] for i in range(len(linhas))]
    save_compact_table(base, dados, linhas, colunas, figsize, first_col_width, fontsize=fontsize)
    pd.DataFrame([{"coeficiente": l.replace("$", ""), "coluna": c.replace("\n", " "), "valor": valores[i][j], "erro_padrao": erros[i][j]}
                  for i, l in enumerate(linhas) for j, c in enumerate(colunas)]).to_csv(base + ".csv", index=False)
    with open(base + ".tex", "w", encoding="utf-8") as fh:
        fh.write("\\begin{tabular}{l" + "c" * len(colunas) + "}\n\\hline\n")
        fh.write("Coef. & " + " & ".join(c.replace("\n", " ") for c in colunas) + " \\\\\n\\hline\n")
        for i, l in enumerate(linhas):
            fh.write(l + " & " + " & ".join(dados[i]) + " \\\\\n")
        fh.write("\\hline\n\\end{tabular}\n")


salvar_tabela_coeficientes("tabela_coeficientes_existencia", [f"$c_{{{i}}}$  " + TEX_TERMOS[i] for i in range(6)], ["Valor"],
                           [[c] for c in COEF_EXIST], [[e] for e in ERRO_EXIST], figsize=(4.6, 4.4), first_col_width=0.45)

### Desempenho no teste e mapa de estados

In [ ]:
y = teste_c["ressonante"].to_numpy(float)
p = p_res(teste_c["u"], teste_c["v"], COEF_EXIST)
xj = y + np.random.default_rng(RANDOM_SEED).uniform(-0.05, 0.05, len(y))
m_ = MET_EXIST["teste"]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(xj[y == 0], p[y == 0], s=14, facecolors="white", edgecolors="#6488A5", linewidths=0.7, zorder=2, label="Sem ressonância")
ax.scatter(xj[y == 1], p[y == 1], s=14, facecolors="#8B1A1A", edgecolors="black", linewidths=0.4, zorder=3, label="Com ressonância")
ax.axhline(0.5, color="black", linestyle=":", linewidth=1.2, zorder=1, label=r"$P_{\mathrm{res}}=0{,}5$")
ax.set_xlim(-0.15, 1.15)
ax.set_ylim(-0.03, 1.03)
ax.set_xticks([0, 1])
ax.set_xticklabels([r"$0$", r"$1$"])
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels([r"$0$", r"$0{,}25$", r"$0{,}50$", r"$0{,}75$", r"$1$"])
estilizar(ax, "Existência observada", r"$P_{\mathrm{res}}$ previsto")
adicionar_caixa(ax, f"$N={m_['N']}$\n" + f"Acurácia $={dec(m_['acuracia'], 3)}$\n" + f"Precisão $={dec(m_['precisao'], 3)}$\n"
                + f"Revocação $={dec(m_['revocacao'], 3)}$\n" + f"ROC AUC $={dec(m_['AUC'], 4)}$", x=0.035, ha="left")
adicionar_legenda(ax)
fig.subplots_adjust(left=0.14, right=0.985, bottom=0.15, top=0.98)
salvar_figura(fig, "grafico_existencia")
plt.show()
plt.close(fig)

for idioma in ("pt", "en"):
    plot_mapa_fase(resumo, COEF_EXIST, idioma)

## 9. Relação de recorrência

### 9.1 Equação característica no limite subcomprimento de onda

As ressonâncias internas ficam próximas da parte real dos zeros de $D_\ell$ (Seção 2),

$$
D_\ell=h_\ell'(x_M)\,j_\ell(x_S)-m_t\,j_\ell'(x_S)\,h_\ell(x_M)=0 .
$$

Para $x_M\ll1$, a função de Hankel é dominada por $y_\ell\propto x_M^{-(\ell+1)}$, de modo que $h_\ell'(x_M)/h_\ell(x_M)\simeq-(\ell+1)/x_M$
(a parte imaginária, $i$ para $\ell=0$, é a perda por radiação e dá a largura do pico). Com $x_M=x_S/m$ e $m_t=m/\rho$, o fator $m$ se cancela:

$$
x\,j_\ell'(x)+\rho\,(\ell+1)\,j_\ell(x)=0
\quad\Longleftrightarrow\quad
x\,j_{\ell-1}(x)=(1-\rho)(\ell+1)\,j_\ell(x),\qquad x=x_S .
$$

As raízes $\zeta_{\ell,s}(\rho)$, $s=1,2,\dots$, dependem só de $\ell$ e de $\rho=\rho_S/\rho_M$, e as ressonâncias ficam em

$$
x_{M;\ell,s}=\frac{\zeta_{\ell,s}(\rho)}{m}.
$$

A razão de velocidades apenas reescala o espectro, e a razão de densidades decide onde cada modo cai. Casos-limite:
$\rho=1$ dá os zeros de $j_{\ell-1}$ (por isso o passo $\pi/m$ e a Eq. da difração); $\rho\to0$ dá os zeros de $j_\ell'$; $\rho\to\infty$, os zeros de $j_\ell$.
Para $\ell=0$ a equação é exata na forma $x\cot x=1-\rho$, e para $\rho\ll1$ sua primeira raiz é $x^2\simeq3\rho$, isto é,
$x_M=\sqrt{3\rho}/m=x_{\mathrm{ref}}$ (Minnaert).

### 9.2 Forma fechada das raízes

Com as formas assintóticas de $j_{\ell-1}$ e $j_\ell$, a equação vira $\tan[x-(\ell-1)\pi/2]=(\rho-1)(\ell+1)/x$, e o ponto de partida é

$$
\zeta^{(0)}_{\ell,s}=z_{\ell-1,s}+\arctan\!\left[\frac{(\rho-1)(\ell+1)}{z_{\ell-1,s}}\right],
\qquad
z_{\ell-1,s}=\beta-\frac{\mu_\ell-1}{8\beta}-\frac{4(\mu_\ell-1)(7\mu_\ell-31)}{3(8\beta)^3},
$$

com $\beta=(s+\ell/2-1/2)\pi$ e $\mu_\ell=(2\ell-1)^2$ (zeros de McMahon). Para o modo monopolar com $\rho<1$, usa-se o aproximante de Padé de
$x\cot x$, $\zeta^{(0)}_{0,1}=\sqrt{15\rho/(5+\rho)}$, que generaliza Minnaert. Duas correções de Newton completam a forma fechada:

$$
\zeta^{(k+1)}=\zeta^{(k)}-\frac{F(\zeta^{(k)})}{F'(\zeta^{(k)})},\qquad
F(x)=x\,j_\ell'+\rho(\ell+1)\,j_\ell,\qquad
F'(x)=\big[\rho(\ell+1)-1\big]j_\ell'-\Big[x-\frac{\ell(\ell+1)}{x}\Big]j_\ell ,
$$

em que $F'$ usa a equação de Bessel esférica para eliminar $j_\ell''$. Nenhum coeficiente é ajustado aos dados.

### 9.3 Recorrência, contagem e existência

Por família de $\ell$, a ressonância seguinte é a próxima raiz, $x_{\ell,n+1}=\zeta_{\ell,s_n+1}(\rho)/m$. Para a sequência inteira de picos de
$W_T$, sem conhecer o $\ell$ de cada pico, a seguinte é a menor raiz acima da atual entre todos os multipolos,

$$
x_{M,n+1}=\min_{0\le\ell\le\ell_{\max}}\Big\{\frac{\zeta_{\ell,s}(\rho)}{m}\ :\ \zeta_{\ell,s}(\rho)>m\,x_{M,n}\Big\}.
$$

O número de ressonâncias até $x_M=1$ é $N=\sum_\ell\#\{s:\zeta_{\ell,s}\le m\}$. Os modos com $\ell\ge1$ perdem energia por radiação
muito devagar ($Q\propto x_M^{-(2\ell+1)}$) e aparecem sempre; o modo $\ell=0$ pode ser largo demais. Mantendo a parte imaginária de
$h_0'/h_0=i-1/x_M$, a equação para $\ell=0$ fica $F(x)=i\rho\,x_M\,j_0(x)$, o que desloca a raiz para o plano complexo e dá

$$
Q_0=\frac{\mathrm{Re}\,x}{2\,|\mathrm{Im}\,x|}=\frac{m\,|F'(\zeta_{0,s})|}{2\rho\,|j_0(\zeta_{0,s})|},
$$

que no limite de Minnaert se reduz a $Q_0=1/x_M$. Um modo $\ell=0$ só aparece como pico de $W_T$ quando $Q_0\ge Q_c$, com
$Q_c=1{,}3$ (limiar observado no banco de dados, equivalente a $x_M\lesssim0{,}77$ para a bolha). A relação de existência pela equação
característica é, então, $N\ge1$.

In [ ]:
def ler_picos():
    partes = pd.read_csv(ARQ_RES, usecols=["global", "xM_peak", "primary_l"], chunksize=2_000_000,
                         dtype={"global": np.int32, "xM_peak": float, "primary_l": float})
    df = pd.concat(partes, ignore_index=True)
    df = df.loc[np.isfinite(df["xM_peak"]) & (df["xM_peak"] >= 1e-4) & (df["xM_peak"] <= 1.0)]
    return df.sort_values(["global", "xM_peak"]).reset_index(drop=True)


RES = ler_picos()
por_curva = RES.groupby("global").size()
completas = por_curva.index[por_curva.to_numpy() == resumo.set_index("global")["N"].reindex(por_curva.index).to_numpy()]
RES = RES.loc[RES["global"].isin(completas)].reset_index(drop=True)
CURVAS = resumo.loc[(resumo["N"] == 0) | resumo["global"].isin(completas)].set_index("global")
PARAM = CURVAS[["m", "rho_ratio", "u", "v"]]
print(f"picos lidos: {len(RES)} em {len(completas)} curvas com todas as posições")

# ---------------- equação característica e forma fechada das raízes ----------------
LMAX_ESP = 10                        # mesmo l_max do banco de dados
Q_C = 1.3                            # modo l = 0 só aparece em W_T se Q_0 >= Q_c


def zeros_mcmahon(l, s):
    # z_{l-1,s}: beta - (mu-1)/(8 beta) - 4(mu-1)(7mu-31)/[3(8 beta)^3]
    beta = (s + 0.5 * l - 0.5) * np.pi
    mu = (2.0 * l - 1.0) ** 2
    return beta - (mu - 1.0) / (8.0 * beta) - 4.0 * (mu - 1.0) * (7.0 * mu - 31.0) / (3.0 * (8.0 * beta) ** 3)


def F_caracteristica(l, rho, x):
    # F = x j_l' + rho (l+1) j_l ;  F' = [rho (l+1) - 1] j_l' - [x - l(l+1)/x] j_l
    j, dj = spherical_jn(l, x), spherical_jn(l, x, derivative=True)
    return x * dj + rho * (l + 1) * j, (rho * (l + 1) - 1.0) * dj - (x - l * (l + 1) / x) * j


def zeta(l, s, rho, n_newton=2):
    # raiz zeta_{l,s}(rho): McMahon + arctan (Padé para o monopolo com rho < 1) e duas correções de Newton
    s, rho = np.broadcast_arrays(np.asarray(s, float), np.asarray(rho, float))
    z = zeros_mcmahon(l, s)
    x = z + np.arctan((rho - 1.0) * (l + 1) / z)
    if l == 0:
        x = np.where((s == 1) & (rho < 1), np.sqrt(15.0 * rho / (5.0 + rho)), x)
    for _ in range(n_newton):
        F, dF = F_caracteristica(l, rho, x)
        x = x - F / dF
    return x


def fator_q_monopolo(m, rho, z):
    # Q_0 = m |F'(zeta)| / [2 rho |j_0(zeta)|], perda por radiação do modo l = 0 (Q_0 = 1/x_M no limite de Minnaert)
    _, dF = F_caracteristica(0, rho, z)
    return m * np.abs(dF) / (2.0 * rho * np.abs(spherical_jn(0, z)))


def raizes_vizinhas(l, rho, xS, n=7):
    # n raízes consecutivas zeta_{l,s} em torno de x_S; o índice s sai da inversão de beta
    rho, xS = np.broadcast_arrays(np.asarray(rho, float), np.asarray(xS, float))
    s0 = np.maximum(1.0, np.floor(xS / np.pi - 0.5 * l + 0.5) - 3.0)
    return np.stack([zeta(l, s0 + d, rho) for d in range(n)])


def raiz_seguinte(Z, xS):
    # identifica a raiz do pico atual (a mais próxima de x_S) e devolve a seguinte
    Z = np.sort(Z, axis=0)
    k = np.abs(Z - xS[None, :]).argmin(axis=0)
    prox = np.take_along_axis(Z, np.minimum(k + 1, len(Z) - 1)[None, :], axis=0)[0]
    return np.where(k + 1 < len(Z), prox, np.nan)


def espectro(m, rho, x_min=1e-4, x_max=1.0, por_l=False):
    # todas as ressonâncias x_M = zeta_{l,s}/m em [x_min, x_max]
    saida = {}
    for l in range(LMAX_ESP + 1):
        s = np.arange(1, int(m * x_max / np.pi + 0.5 * l + 3) + 1)
        x = zeta(l, s, rho) / m
        if l == 0:
            x = x[fator_q_monopolo(m, rho, x * m) >= Q_C]
        saida[l] = x[(x >= x_min) & (x <= x_max)]
    return saida if por_l else np.sort(np.concatenate(list(saida.values())))


# validação 1: posição de cada pico observado contra a raiz mais próxima da sua família
erros, xs_val = [], []
for (g, l), d in RES.loc[np.isfinite(RES["primary_l"])].groupby(["global", "primary_l"]):
    m, rho = PARAM.at[g, "m"], PARAM.at[g, "rho_ratio"]
    raizes = espectro(m, rho, x_min=0, x_max=1.2, por_l=True)[int(l)]
    if l == 0:                                     # inclui o monopolo largo na comparação de posição
        raizes = np.sort(np.r_[zeta(0, 1, rho) / m, raizes])
    x = d["xM_peak"].to_numpy(float)
    k = np.abs(raizes[None, :] - x[:, None]).argmin(axis=1)
    erros.append(100 * np.abs(raizes[k] - x) / x); xs_val.append(m * x)
ERRO_POS, XS_POS = np.concatenate(erros), np.concatenate(xs_val)
for nome, mk in (("esfera pequena", XS_POS < np.pi), ("esfera grande", XS_POS >= np.pi)):
    print(f"posição, {nome}: {mk.sum()} picos | erro mediano {np.median(ERRO_POS[mk]):.2g} % | "
          f"percentil 99 {np.quantile(ERRO_POS[mk], 0.99):.2g} % | máximo {ERRO_POS[mk].max():.2g} %")

# validação 2: contagem e existência em todas as curvas do banco
t0 = time.perf_counter()
resumo["N_esp"] = [len(espectro(m, r)) for m, r in zip(resumo["m"], resumo["rho_ratio"])]
T_ESPECTRO = (time.perf_counter() - t0) / len(resumo)
exist_esp = resumo["N_esp"] > 0
print(f"existência pelo espectro: acerto {np.mean(exist_esp == (resumo['N'] > 0)):.4f} | "
      f"FP {int((exist_esp & (resumo['N'] == 0)).sum())} | FN {int((~exist_esp & (resumo['N'] > 0)).sum())}")
for faixa, mk in (("m <= 10", resumo["m"] <= 10), ("m > 10", resumo["m"] > 10)):
    r = resumo.loc[mk & (resumo["N"] > 0)]
    print(f"contagem, {faixa}: N previsto = N observado em {np.mean(r['N_esp'] == r['N']):.3f} das curvas ressonantes")
if "N_res_polos_area" in resumo:
    print(f"N previsto = número de polos em {np.mean(resumo['N_esp'] == resumo['N_res_polos_area']):.3f} das curvas")
print(f"tempo do espectro completo: {T_ESPECTRO * 1e3:.2g} ms por configuração")

### Espectro universal

Como $x_S=m\,x_M=\zeta_{\ell,s}(\rho)$, os picos de todas as curvas, com qualquer $c_S/c_M$, caem sobre as mesmas curvas quando
desenhados em $x_S$ contra $\rho_S/\rho_M$.

In [ ]:
RHO_G = np.logspace(-4, np.log10(3000), 500)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, l in zip(axes, (0, 1)):
    d = RES.loc[RES["primary_l"] == l].join(PARAM, on="global")
    xs = (d["m"] * d["xM_peak"]).to_numpy()
    ok = xs <= 11
    ax.scatter(d["rho_ratio"].to_numpy()[ok], xs[ok], s=10, facecolors="white", edgecolors="0.45", linewidths=0.5,
               zorder=1, rasterized=True, label=r"Picos de $W_T$ (todas as $c_S/c_M$)")
    for s in range(1, 5):
        ax.plot(RHO_G, zeta(l, s, RHO_G), color="#8B1A1A", linewidth=1.5, zorder=3,
                label="Forma fechada" if s == 1 else None)
    if l == 0:
        rm = RHO_G[RHO_G <= 10]
        ax.plot(rm, np.sqrt(3 * rm), color="#6488A5", linestyle="--", linewidth=1.5, zorder=2,
                label=r"Minnaert, $\sqrt{3\rho_S/\rho_M}$")
    ax.set_xscale("log")
    ax.set_xlim(1e-4, 3000); ax.set_ylim(0, 11)
    estilizar(ax, r"$\rho_S/\rho_M$", r"$x_S=m\,x_M$" if l == 0 else None)
    adicionar_caixa(ax, rf"$\ell={l}$", x=0.035, ha="left")
h, r = axes[0].get_legend_handles_labels()
leg = axes[1].legend(h, r, loc="lower right", fontsize=8, frameon=True)
leg.get_frame().set(edgecolor="black", facecolor="white", linewidth=1.0)
fig.subplots_adjust(left=0.07, right=0.985, bottom=0.16, top=0.98, wspace=0.06)
salvar_figura(fig, "grafico_espectro_universal")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(resumo["c_razao"], resumo["N"], s=10, facecolors="white", edgecolors="0.45", linewidths=0.5, zorder=2,
           label=r"Picos de $W_T$ observados")
ax.scatter(resumo["c_razao"], resumo["N_esp"], s=5, facecolors="#8B1A1A", edgecolors="none", zorder=3,
           label="Equação característica")
ax.set_xscale("log"); ax.set_yscale("symlog", linthresh=1)
ax.set_ylim(bottom=0)
estilizar(ax, r"$c_S/c_M$", r"$N$ até $x_M=1$")
adicionar_legenda(ax, loc="upper right")
fig.subplots_adjust(left=0.15, right=0.985, bottom=0.15, top=0.98)
salvar_figura(fig, "grafico_contagem_espectro")
plt.show()
plt.close(fig)

### 9.4 Comparação com as formas ajustadas

As formas usadas antes, com coeficientes ajustados por mínimos quadrados nas curvas de treino, ficam como comparação:

$$
x_{M,n+1}=\Big[a_0+a_1\ln m+a_2\ln\rho+a_3(\ln m)^2+a_4\ln m\ln\rho+a_5(\ln\rho)^2\Big]\,\chi
+\Big[b_0+b_1\ln m+\dots+b_5(\ln\rho)^2\Big]\,x_{M,n},
$$

com âncora $\chi=x_{\mathrm{ref}}$ (forma anterior da tese) ou $\chi=\pi/m$, e o passo de McMahon com um coeficiente,
$x_{\ell,n+1}=x_{\ell,n}+S[z_{\ell-1,s+1}-z_{\ell-1,s}]/m$. Todas são avaliadas nas mesmas curvas de teste: $R^2$ e MAPE de $x_{M,n+1}$,
erro relativo do passo $x_{M,n+1}-x_{M,n}$, desvio da sequência gerada a partir da primeira ressonância e erro no número de ressonâncias até
$x_M=1$, separando esfera pequena ($x_S=m\,x_{M,n}<\pi$) e esfera grande ($x_S\ge\pi$).

In [ ]:
def passo_mcmahon(l, m, x_n):
    s = np.maximum(1.0, np.floor(m * x_n / np.pi - 0.5 * l + 1.0))
    return (zeros_mcmahon(l, s + 1) - zeros_mcmahon(l, s)) / m


class Modelo:
    # "ref": x_{n+1} = x_ref A + B x_n ; "esp": x_{n+1} = (pi/m) A + B x_n ; "mcm": passo de McMahon (1 coef.)
    # "car": equação característica, sem coeficientes ajustados
    def __init__(self, tipo, alvo):
        self.tipo, self.alvo = tipo, alvo

    def matriz(self, m, rho, u, v, x_n):
        anc = x_ref(m, rho) if self.tipo == "ref" else np.pi / m
        T = termos_existencia(u, v)
        return np.column_stack([anc[:, None] * T, np.asarray(x_n)[:, None] * T])

    def ajusta(self, tr):
        if self.tipo == "car":
            self.coef, self.erro = np.array([]), np.array([])
            return self
        m, rho, u, v, xn, xn1 = (tr[k].to_numpy(float) for k in ("m", "rho_ratio", "u", "v", "x_n", "x_np1"))
        if self.tipo == "mcm":
            q = (xn1 - xn) / passo_mcmahon(tr["l"].to_numpy(float), m, xn)
            q = q[np.isfinite(q)]
            boot = [np.median(np.random.default_rng(RANDOM_SEED + k).choice(q, len(q))) for k in range(200)]
            self.coef, self.erro = np.array([np.median(q)]), np.array([np.std(boot)])
            return self
        X = self.matriz(m, rho, u, v, xn)
        self.coef = np.linalg.lstsq(X, xn1, rcond=None)[0]
        res = xn1 - X @ self.coef
        self.erro = np.sqrt(np.clip(np.diag(res @ res / max(len(xn1) - 12, 1) * np.linalg.pinv(X.T @ X)), 0, None))
        return self

    def proximo(self, m, rho, u, v, x_n, l=None):
        m, rho, u, v, x_n = (np.atleast_1d(np.asarray(a, float)) for a in (m, rho, u, v, x_n))
        if self.tipo == "mcm":
            return x_n + self.coef[0] * passo_mcmahon(np.atleast_1d(np.asarray(l, float)), m, x_n)
        if self.tipo == "car":
            xS = m * x_n
            if self.alvo == "wt":                    # sequência de W_T: raízes de todos os l juntas
                Z = np.concatenate([raizes_vizinhas(L, rho, xS) for L in range(LMAX_ESP + 1)])
                return raiz_seguinte(Z, xS) / m
            l = np.atleast_1d(np.asarray(l, float))
            out = np.full(len(xS), np.nan)
            for L in np.unique(l[np.isfinite(l)]).astype(int):
                k = l == L
                out[k] = raiz_seguinte(raizes_vizinhas(L, rho[k], xS[k]), xS[k])
            return out / m
        return self.matriz(m, rho, u, v, x_n) @ self.coef


def transicoes(df, por_familia):
    if por_familia:
        df = df.loc[np.isfinite(df["primary_l"])].sort_values(["global", "primary_l", "xM_peak"])
    g, x, l = df["global"].to_numpy(), df["xM_peak"].to_numpy(float), df["primary_l"].to_numpy(float)
    mesmo = g[1:] == g[:-1]
    if por_familia:
        mesmo &= l[1:] == l[:-1]
    tr = pd.DataFrame({"global": g[:-1][mesmo], "l": l[:-1][mesmo], "x_n": x[:-1][mesmo], "x_np1": x[1:][mesmo]}).join(PARAM, on="global")
    tr["x_S"] = tr["m"] * tr["x_n"]
    tr["teste"] = tr["global"].isin(GLOBAIS_TESTE)
    return tr


TR = {"wt": transicoes(RES, False), "fam": transicoes(RES, True)}
CHAVES = [("wt", "ref"), ("wt", "esp"), ("fam", "esp"), ("fam", "mcm"), ("wt", "car"), ("fam", "car")]
MODELOS = {(alvo, tipo): Modelo(tipo, alvo).ajusta(TR[alvo].loc[~TR[alvo]["teste"]]) for alvo, tipo in CHAVES}
NOMES = {("wt", "ref"): r"$W_T$, $x_{\mathrm{ref}}$ (forma anterior)", ("wt", "esp"): r"$W_T$, $\pi/m$",
         ("fam", "esp"): r"Família $\ell$, $\pi/m$", ("fam", "mcm"): r"Família $\ell$, McMahon",
         ("wt", "car"): r"$W_T$, equação característica", ("fam", "car"): r"Família $\ell$, equação característica"}
ESTILO = {("wt", "ref"): dict(color="#6488A5", linestyle="-", linewidth=2.0),
          ("wt", "esp"): dict(color="#6488A5", linestyle="--", linewidth=1.75),
          ("fam", "esp"): dict(color="0.45", linestyle="-.", linewidth=1.75),
          ("fam", "mcm"): dict(color="black", linestyle=":", linewidth=1.8),
          ("wt", "car"): dict(color="#8B1A1A", linestyle="--", linewidth=2.0),
          ("fam", "car"): dict(color="#8B1A1A", linestyle="-", linewidth=2.0)}
print("S (McMahon) =", MODELOS[("fam", "mcm")].coef[0], "±", MODELOS[("fam", "mcm")].erro[0])

In [ ]:
def r2_mape(obs, pred):
    ok = np.isfinite(obs) & np.isfinite(pred)
    obs, pred = obs[ok], pred[ok]
    if len(obs) < 2:
        return np.nan, np.nan
    return 1 - np.sum((obs - pred) ** 2) / np.sum((obs - obs.mean()) ** 2), 100 * np.mean(np.abs((obs - pred) / obs))


def sequencias(mod, ini):
    # x_{k+1} = f(x_k) a partir de x_0 até passar de x_M = 1, para todas as sequências de uma vez
    m, rho, u, v, l = (ini[k].to_numpy(float) for k in ("m", "rho_ratio", "u", "v", "l"))
    lim, atual = ini["N_lim"].to_numpy(int), ini["x0"].to_numpy(float).copy()
    tam, ativo = np.ones(len(atual), int), np.ones(len(atual), bool)
    bi, bx = [np.arange(len(atual))], [atual.copy()]
    while ativo.any():
        idx = np.flatnonzero(ativo)
        with np.errstate(all="ignore"):
            nxt = mod.proximo(m[idx], rho[idx], u[idx], v[idx], atual[idx], l[idx])
        ok = np.isfinite(nxt) & (nxt > atual[idx]) & (nxt <= 1.0)
        bi.append(idx[ok]); bx.append(nxt[ok])
        atual[idx[ok]] = nxt[ok]
        tam[idx[ok]] += 1
        ativo[idx[~ok]] = False
        ativo &= tam < lim
    I, Xs = np.concatenate(bi), np.concatenate(bx)
    o = np.lexsort((Xs, I))
    return np.split(Xs[o], np.cumsum(np.bincount(I, minlength=len(atual)))[:-1])


TESTE_RES = RES.loc[RES["global"].isin(GLOBAIS_TESTE)]


def avaliar(chave):
    alvo, _ = chave
    mod, te = MODELOS[chave], TR[alvo].loc[TR[alvo]["teste"]]
    pred = mod.proximo(te["m"], te["rho_ratio"], te["u"], te["v"], te["x_n"], te["l"])
    obs, xn = te["x_np1"].to_numpy(float), te["x_n"].to_numpy(float)
    passo = obs - xn
    erro_passo = np.where(passo > 0, np.abs((pred - xn) - passo) / np.where(passo > 0, passo, 1), np.nan)
    peq = (te["x_S"] < np.pi).to_numpy()
    out = {"x_n": xn, "obs": obs, "pred": pred, "erro_passo": erro_passo, "x_S": te["x_S"].to_numpy(float), "m": te["m"].to_numpy(float)}
    for g, mk in (("pequena", peq), ("grande", ~peq), ("todas", np.ones_like(peq))):
        out[f"R2_{g}"], out[f"MAPE_{g}"] = r2_mape(obs[mk], pred[mk])
        e = erro_passo[mk][np.isfinite(erro_passo[mk])]
        out[f"passo_{g}"] = 100 * np.median(e) if len(e) else np.nan
        out[f"n_{g}"] = int(mk.sum())
    df = TESTE_RES if alvo == "wt" else TESTE_RES.loc[np.isfinite(TESTE_RES["primary_l"])]
    grupos = df.groupby(["global"] if alvo == "wt" else ["global", "primary_l"], sort=True)["xM_peak"]
    obs_seq = [np.sort(g.to_numpy(float)) for _, g in grupos]
    ini = grupos.first().reset_index().rename(columns={"xM_peak": "x0", "primary_l": "l"})
    if "l" not in ini:
        ini["l"] = np.nan
    ini = ini.join(PARAM, on="global")
    ini["N_lim"] = [4 * len(o) + 50 for o in obs_seq]
    desv, xs_desv, m_desv, n_pred = [], [], [], {}
    for g, o, s in zip(ini["global"], obs_seq, sequencias(mod, ini)):
        n = min(len(o), len(s))
        if n >= 2:
            desv.append(np.abs(s[1:n] - o[1:n]) / o[1:n])
            xs_desv.append(PARAM.at[g, "m"] * o[1:n])
            m_desv.append(np.full(n - 1, PARAM.at[g, "m"]))
        n_pred[g] = n_pred.get(g, 0) + len(s)
    desv = np.concatenate(desv) if desv else np.array([])
    xs_desv = np.concatenate(xs_desv) if xs_desv else np.array([])
    m_desv = np.concatenate(m_desv) if m_desv else np.array([])
    for g, mk in (("pequena", xs_desv < np.pi), ("grande", xs_desv >= np.pi), ("todas", np.ones(len(desv), bool)),
                  ("m10", m_desv <= 10)):
        out[f"seq_{g}"] = 100 * np.median(desv[mk]) if mk.any() else np.nan
    cont = pd.DataFrame({"global": list(n_pred), "N_pred": list(n_pred.values())}).join(CURVAS[["N", "m"]], on="global")
    cont["erro_N"] = 100 * np.abs(cont["N_pred"] - cont["N"]) / cont["N"]
    out["N_m10"] = float(cont.loc[cont["m"] <= 10, "erro_N"].median())
    if "N_res_polos_area" in CURVAS:                 # contagem física (polos de D_l), sem o limite da malha
        polos = CURVAS["N_res_polos_area"].reindex(cont["global"]).to_numpy(float)
        ok = polos > 0
        out["N_polos"] = float(np.median(100 * np.abs(cont["N_pred"].to_numpy()[ok] - polos[ok]) / polos[ok]))
    out["contagem"] = cont
    for g, mk in (("pequena", cont["m"] < np.pi), ("grande", cont["m"] >= np.pi), ("todas", cont["m"] > 0)):
        out[f"N_{g}"] = float(cont.loc[mk, "erro_N"].median()) if mk.any() else np.nan
    return out


AVAL = {chave: avaliar(chave) for chave in MODELOS}
NOMES_TEXTO = {("wt", "ref"): "W_T, x_ref (anterior)", ("wt", "esp"): "W_T, pi/m", ("fam", "esp"): "família l, pi/m",
               ("fam", "mcm"): "família l, McMahon", ("wt", "car"): "W_T, eq. característica",
               ("fam", "car"): "família l, eq. característica"}
tabela = pd.DataFrame({NOMES_TEXTO[k]: {
    c: a[c] for c in ("R2_todas", "MAPE_todas", "passo_pequena", "passo_grande", "seq_grande", "seq_m10", "N_todas", "N_m10", "N_polos")} for k, a in AVAL.items()}).T
print(tabela.round(4).to_string())
tabela.to_csv(os.path.join(TAB_DIR, "tabela_metricas_recorrencia.csv"))

### Coeficientes das formas ajustadas

A equação característica não tem coeficientes ajustados. As tabelas abaixo guardam os coeficientes das formas usadas na comparação, e a
tabela da tese passa a reunir apenas os coeficientes $c_i$ da relação de existência.

In [ ]:
for alvo, nome, tipos in (("wt", "picos_WT", ("ref", "esp")), ("fam", "familias_l", ("esp",))):
    colunas, valores, erros = [], [[] for _ in range(6)], [[] for _ in range(6)]
    for tipo, anc in ((t, {"ref": r"$x_{\mathrm{ref}}$", "esp": r"$\pi/m$"}[t]) for t in tipos):
        for parte, d in (("A", 0), ("B", 6)):
            colunas.append(f"Âncora {anc}\n${parte}$")
            for i in range(6):
                valores[i].append(MODELOS[(alvo, tipo)].coef[d + i])
                erros[i].append(MODELOS[(alvo, tipo)].erro[d + i])
    salvar_tabela_coeficientes(f"tabela_coeficientes_recorrencia_{nome}", [f"${i}$" for i in range(6)], colunas, valores, erros,
                               figsize=(12.5 if len(tipos) == 2 else 7.5, 4.8), first_col_width=0.08 if len(tipos) == 2 else 0.14, fontsize=16)

with open(os.path.join(TAB_DIR, "tabela_tese_existencia.tex"), "w", encoding="utf-8") as fh:
    fh.write("\\begin{tabular}{lc}\n\\toprule\n\\textbf{Termo} & $c_i$ \\\\\n\\midrule\n")
    for i in range(6):
        fh.write(f"{TEX_TERMOS[i]} & {fmt_pm(COEF_EXIST[i], ERRO_EXIST[i])} \\\\\n")
    fh.write("\\bottomrule\n\\end{tabular}\n")
print(open(os.path.join(TAB_DIR, "tabela_tese_existencia.tex"), encoding="utf-8").read())

### Gráficos da recorrência

Posições previstas contra observadas (escala log), distribuição do erro do passo, métricas por regime e erro em função de $c_S/c_M$.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, chave in zip(axes.ravel(), [("wt", "ref"), ("fam", "mcm"), ("wt", "car"), ("fam", "car")]):
    a = AVAL[chave]
    ax.scatter(a["x_n"], a["obs"], s=10, facecolors="white", edgecolors="0.45", linewidths=0.5, zorder=1,
               label="Observado", rasterized=True)
    ax.scatter(a["x_n"], a["pred"], s=6, facecolors="#6488A5", edgecolors="none", zorder=2, label="Previsto", rasterized=True)
    lim = [1e-4, 1.0]
    ax.plot(lim, lim, color="black", linestyle="--", linewidth=1.0, zorder=3, label="1:1")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(*lim); ax.set_ylim(*lim)
    estilizar(ax, r"$x_{M,n}$", r"$x_{M,n+1}$")
    adicionar_caixa(ax, NOMES[chave] + "\n" + rf"$R^2={dec(a['R2_todas'], 4)}$" + "\n"
                    + (rf"Erro do passo $={dec(a['passo_todas'], 2)}${PCT}" if a["passo_todas"] >= 0.01
                       else rf"Erro do passo $<0{{,}}01${PCT}"), x=0.035, ha="left")
    adicionar_legenda(ax)
fig.subplots_adjust(left=0.09, right=0.985, bottom=0.09, top=0.98, wspace=0.25, hspace=0.27)
salvar_figura(fig, "grafico_recorrencia_dispersao_2x2")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, grupo in zip(axes, ("pequena", "grande")):
    for chave, est in ESTILO.items():
        a = AVAL[chave]
        mk = a["x_S"] < np.pi if grupo == "pequena" else a["x_S"] >= np.pi
        e = np.sort(100 * a["erro_passo"][mk])
        e = e[np.isfinite(e) & (e > 0)]
        if len(e):
            ax.plot(e, np.arange(1, len(e) + 1) / len(e), label=NOMES[chave], **est)
    ax.set_xscale("log")
    ax.set_ylim(0, 1.02)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: dec(v, 2)))
    estilizar(ax, f"Erro relativo do passo ({PCT})", "Fração das transições" if grupo == "pequena" else None)
    adicionar_caixa(ax, r"Esfera pequena, $x_S<\pi$" if grupo == "pequena" else r"Esfera grande, $x_S\geq\pi$", x=0.035, ha="left")
adicionar_legenda(axes[0], loc="upper left", fontsize=8, bbox_to_anchor=(0.01, 0.86))
fig.subplots_adjust(left=0.08, right=0.985, bottom=0.16, top=0.98, wspace=0.08)
salvar_figura(fig, "grafico_erro_do_passo")
plt.show()
plt.close(fig)

In [ ]:
HACHURA = {("wt", "ref"): None, ("wt", "esp"): "...", ("fam", "esp"): "///", ("fam", "mcm"): None,
           ("wt", "car"): None, ("fam", "car"): "///"}
metricas = [("R2", r"$R^2$ de $x_{M,n+1}$", False), ("MAPE", f"MAPE de $x_{{M,n+1}}$ ({PCT})", True),
            ("passo", f"Erro mediano do passo ({PCT})", True), ("seq", f"Desvio mediano da sequência ({PCT})", True),
            ("N", f"Erro mediano em $N$ ({PCT})", True)]
fig, axes = plt.subplots(2, 3, figsize=(10, 6.5))
axes = axes.ravel()
largura = 0.8 / len(MODELOS)
for ax, (nome, rot, log) in zip(axes, metricas):
    for k, chave in enumerate(MODELOS):
        vals = np.array([AVAL[chave][f"{nome}_{g}"] for g in ("pequena", "grande")], float)
        pos = np.arange(2) + (k - (len(MODELOS) - 1) / 2) * largura
        ax.bar(pos, np.nan_to_num(vals), width=largura, color=ESTILO[chave]["color"], edgecolor="black", linewidth=0.6,
               hatch=HACHURA[chave], label=NOMES[chave], zorder=3)
        for x_, v_ in zip(pos, vals):
            if not np.isfinite(v_) or (log and v_ == 0):
                ax.text(x_, 0, "--" if not np.isfinite(v_) else "0", ha="center", va="bottom", fontsize=7)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([r"Pequena, $x_S<\pi$", r"Grande, $x_S\geq\pi$"])
    if log:
        ax.set_yscale("symlog", linthresh=1e-4)
        ax.set_ylim(bottom=0)
    else:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: dec(v, 2)))
    estilizar(ax)
    ax.set_title(rot, fontsize=10)
axes[-1].axis("off")
h, l = axes[0].get_legend_handles_labels()
leg = axes[-1].legend(h, l, loc="center", fontsize=9, frameon=True)
leg.get_frame().set(edgecolor="black", facecolor="white", linewidth=1.0)
fig.subplots_adjust(left=0.07, right=0.985, bottom=0.07, top=0.95, wspace=0.28, hspace=0.38)
salvar_figura(fig, "grafico_metricas_pequena_grande_pt")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))
for chave, est in ESTILO.items():
    a = AVAL[chave]
    med = pd.DataFrame({"c": 1 / a["m"], "e": 100 * a["erro_passo"]}).dropna().groupby("c")["e"].median()
    axes[0].plot(med.index, med.to_numpy(), marker="o", markersize=3, label=NOMES[chave], **est)
    c = a["contagem"].assign(c=1 / a["contagem"]["m"]).groupby("c")["erro_N"].mean()
    axes[1].plot(c.index, c.to_numpy(), marker="o", markersize=3, **est)
axes[0].set_yscale("log")
axes[1].set_yscale("symlog", linthresh=1e-1)
axes[1].set_ylim(bottom=0)
for ax, rot in zip(axes, (f"Erro mediano do passo ({PCT})", f"Erro médio em $N$ ({PCT})")):
    ax.set_xscale("log")
    ax.axvline(1 / np.pi, color="0.45", linestyle=":", linewidth=1.0)
    estilizar(ax, r"$c_S/c_M$", rot)
h, r = axes[0].get_legend_handles_labels()
leg = fig.legend(h, r, loc="lower center", ncol=3, fontsize=8, frameon=True, bbox_to_anchor=(0.5, 0.0))
leg.get_frame().set(edgecolor="black", facecolor="white", linewidth=1.0)
fig.subplots_adjust(left=0.08, right=0.985, bottom=0.27, top=0.98, wspace=0.25)
salvar_figura(fig, "grafico_erro_contra_cS_cM")
plt.show()
plt.close(fig)

## 10. Custo computacional

Tempo do cálculo completo por configuração (`runtime_s` do gerador, na máquina onde o banco de dados rodou) contra o tempo das relações
medido aqui: a relação de existência para todas as configurações de uma vez e, para cada curva de teste, a sequência inteira de
ressonâncias até $x_M=1$ (todos os $\ell$) dada pela equação característica na forma fechada.

In [ ]:
t0 = time.perf_counter()
for _ in range(5):
    p_res(resumo["u"].to_numpy(), resumo["v"].to_numpy(), COEF_EXIST)
t_exist = (time.perf_counter() - t0) / 5 / len(resumo)

custo = []
for g in TESTE_RES["global"].unique():
    m, rho = CURVAS.at[g, "m"], CURVAS.at[g, "rho_ratio"]
    t0 = time.perf_counter()
    espectro(m, rho)                     # todas as ressonâncias até x_M = 1, todos os l
    custo.append({"c": 1 / m, "completo": CURVAS.at[g, "runtime_s"], "relacoes": time.perf_counter() - t0 + t_exist})
custo = pd.DataFrame(custo)
print(f"cálculo completo: mediana {resumo['runtime_s'].median():.3g} s | relações: mediana {custo['relacoes'].median():.3g} s "
      f"| ganho mediano {np.median(custo['completo'] / custo['relacoes']):.3g}x")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(resumo["c_razao"], resumo["runtime_s"], s=8, facecolors="white", edgecolors="0.45", linewidths=0.5,
           label="Cálculo completo", zorder=2)
ax.scatter(custo["c"], custo["relacoes"], s=10, facecolors="#8B1A1A", edgecolors="black", linewidths=0.3,
           label="Relações", zorder=3)
ax.axhline(t_exist, color="black", linestyle=":", linewidth=1.0, label="Só a existência")
ax.set_xscale("log"); ax.set_yscale("log")
estilizar(ax, r"$c_S/c_M$", "Tempo por configuração (s)")
adicionar_legenda(ax, loc="lower left", bbox_to_anchor=(0.02, 0.12))
fig.subplots_adjust(left=0.16, right=0.985, bottom=0.15, top=0.98)
salvar_figura(fig, "grafico_custo")
plt.show()
plt.close(fig)

## 11. Arquivos gerados

Lista o que foi gravado em `resultados/` e cria `resultados_esfera_fluida.zip` ao lado da pasta (no Colab, dentro de
`Meu Drive/sphere_final/`), pronto para baixar ou para subir no GitHub.

In [ ]:
for sub in (FIG_DIR, TAB_DIR):
    arqs = sorted(os.listdir(sub))
    nomes = sorted({os.path.splitext(a)[0] for a in arqs})
    print(f"{sub}: {len(arqs)} arquivos, {len(nomes)} itens")
    for n in nomes:
        print("   ", n)
zip_path = shutil.make_archive(os.path.join(os.path.dirname(SAIDA), "resultados_esfera_fluida"), "zip",
                               root_dir=os.path.dirname(SAIDA), base_dir=os.path.basename(SAIDA))
print("zip:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
if NO_COLAB:
    print("No Colab os arquivos já estão no Drive; a sincronização pode levar alguns minutos para aparecer no navegador.")